# 📗 Cypher 심화: 다중 조건·WITH 파이프라인

앞 시간에는 관계를 이어 **경로**를 찾았습니다. 이번에는 찾은 것들을 **여러 조건으로 거르고, 단계별로 다듬고, 정렬해 상위만** 뽑습니다. 맛집 데이터로 "평점 4.5 이상이면서 5만 원 이하인 집", "평점 상위 두 곳의 셰프" 같은 추천 쿼리를 한 조건씩 쌓아 올립니다.

## ⏪ 복습: 지난 시간까지

- **가변길이 `*1..3`·`shortestPath`**: 관계를 여러 칸 이어 경로를 찾았습니다.
- **`length`·`nodes`·`relationships`**: 경로를 해부해 이동 수·지나는 노드·탄 관계를 꺼냈습니다.
- **리스트 표현식 `[n IN nodes(p) | n.name]`·`all`·`any`·`none`**: 목록에서 값만 뽑고, 경로가 지난 구간 전부에 조건을 걸었습니다.
- **`WHERE`(지난 단원)**: `=`·`<>`·`AND`·`OR`·`NOT`·`IS NULL` 로 걸렀고, `DISTINCT`·`ORDER BY`·`LIMIT` 으로 결과를 다듬었습니다. 오늘은 여기에 **`IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH`** 와 **정규식**을 더합니다. 새 연산자도 지난 시간과 똑같이 `AND`·`OR` 로 묶어 씁니다.

**오늘의 목표**

**1. 여러 방식으로 거르기**
- [ ] (1-1) **`IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH`** 로 거른다.
- [ ] (1-2) **정규식 `=~`** 로 여러 패턴을 한 번에 쓰고, **목록을 파라미터**로 넘겨 거른다.

**2. 관계의 있고 없음**
- [ ] (2-1) 한 패턴에 **관계 여러 종류**를 이어 붙여 두 단계 이상을 한 번에 묻는다.
- [ ] (2-2) **관계가 있는지·없는지 자체**를 조건으로 쓴다(`WHERE (a)-[:R]->(b)`).
- [ ] (2-3) **`EXISTS { }`** 로 그 조건을 더 넓게 쓴다.

**3. OPTIONAL MATCH**
- [ ] (3-1~3-2) **`OPTIONAL MATCH`** 로 짝이 없는 노드도 빠뜨리지 않는다(없는 자리는 `null`).
- [ ] (3-3) 두 노드가 **닿는지 판별**하고, **화살표**로 질문의 뜻이 달라지는 것을 구분한다.

**4. WITH 파이프라인**
- [ ] (4-1~4-3) **`WITH`** 로 값을 넘기고, **새로 만든 값으로 거르고**, 중간에서 정렬해 자른다.

**5. 정렬과 쪽 넘기기**
- [ ] (5-1~5-2) **`ORDER BY`·`LIMIT`·`SKIP`** 으로 정렬해 상위를 뽑고 쪽을 넘긴다.

아래 준비 셀을 먼저 실행하세요. 연결 → 초기화 → 시드 적재 순서입니다.

> ⚠️ **연결 셀이 오류로 멈춘다면** 앞 시간과 같습니다. `ServiceUnavailable` 이면 실습 전용 Neo4j 인스턴스가 꺼져 있는 것이고, `AuthError` 면 `.env` 의 비밀번호가 다른 것이며, 아무 값도 못 읽는 것 같으면 이 폴더에 `.env` 가 없는 것입니다(`.env.example` 을 복사해 만드세요).

In [ ]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

> ⚠️ **아래 초기화 셀은 연결된 데이터베이스의 노드를 전부 지웁니다.** 앞 단원에서 만든 그래프(day28 의 Movies 예제, day29 의 과제 결과)도 함께 사라집니다. 되돌릴 수 없으니 `.env` 가 **실습 전용 DB** 를 가리키는지 먼저 확인하세요.

In [ ]:
# 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요! 노드·관계를 전부 지웁니다.
# MATCH (n) 은 모든 노드, DETACH 는 붙은 관계까지 함께 지우라는 뜻입니다.
run_cypher("MATCH (n) DETACH DELETE n")
print("초기화 완료. 남은 노드:", len(run_cypher("MATCH (n) RETURN n")))

오늘 만들고 조회할 그래프의 전체 모습입니다. 식당 6곳을 가운데 두고 셰프 4명·손님 3명·요리 4가지·지역 3곳이 네 종류의 관계로 이어져 있습니다. 관계가 많아 두 장으로 나눠 그렸습니다. 먼저 **사람과 식당**입니다.

<img src="images/식당_그래프_사람.png" width="820">

이어서 식당이 가리키는 **요리와 지역**입니다. 같은 식당 6곳을 왼쪽·오른쪽 판에 한 번씩 그렸습니다.

<img src="images/식당_그래프_분류.png" width="820">

In [ ]:
# 맛집 추천 시드 적재: 이 셀은 실행만 하세요(그래프를 처음부터 만듭니다).
# CREATE 를 일곱 덩이로 나눠 적었다. 노드를 먼저 만들고
# 요리·지역 관계, 셰프, 손님, 방문 관계를 차례로 잇는다.
# 앞 CREATE 에서 지은 변수를 뒤에서 다시 써서 관계를 잇는다.
# 일부러 심어 둔 둘: 국밥천국·라멘야마에는 셰프가 없고(OPTIONAL MATCH 용),
# 평점 4.6 인 식당이 두 곳이다(동점이 쪽 넘기기를 망치는 것을 보는 용도).
run_cypher("""
CREATE (gangnam:Area {name:'강남'}),
       (hongdae:Area {name:'홍대'}),
       (jongno:Area {name:'종로'})
CREATE (jpn:Cuisine {name:'일식'}),
       (western:Cuisine {name:'양식'}),
       (korean:Cuisine {name:'한식'}),
       (chinese:Cuisine {name:'중식'})
CREATE (sushi:Restaurant {name:'스시효', rating:4.8, price:80000}),
       (pasta:Restaurant {name:'파스타부오노', rating:4.6, price:30000}),
       (gukbap:Restaurant {name:'국밥천국', rating:4.2, price:9000}),
       (dimsum:Restaurant {name:'딤섬각', rating:4.6, price:25000}),
       (ramen:Restaurant {name:'라멘야마', rating:4.3, price:12000}),
       (bistro:Restaurant {name:'비스트로홍', rating:4.7, price:45000})
CREATE (sushi)-[:SERVES]->(jpn), (sushi)-[:LOCATED_IN]->(gangnam),
       (pasta)-[:SERVES]->(western), (pasta)-[:LOCATED_IN]->(hongdae),
       (gukbap)-[:SERVES]->(korean), (gukbap)-[:LOCATED_IN]->(jongno),
       (dimsum)-[:SERVES]->(chinese), (dimsum)-[:LOCATED_IN]->(gangnam),
       (ramen)-[:SERVES]->(jpn), (ramen)-[:LOCATED_IN]->(hongdae),
       (bistro)-[:SERVES]->(western), (bistro)-[:LOCATED_IN]->(gangnam)
CREATE (kimhyo:Chef {name:'김효'})-[:WORKS_AT]->(sushi),
       (leebuono:Chef {name:'이부오노'})-[:WORKS_AT]->(pasta),
       (wanggak:Chef {name:'왕각'})-[:WORKS_AT]->(dimsum),
       (hongbi:Chef {name:'홍비'})-[:WORKS_AT]->(bistro)
CREATE (jimin:Diner {name:'지민'}),
       (hajun:Diner {name:'하준'}),
       (seoyeon:Diner {name:'서연'})
CREATE (jimin)-[:VISITED]->(sushi), (jimin)-[:VISITED]->(dimsum), (jimin)-[:VISITED]->(bistro),
       (hajun)-[:VISITED]->(pasta), (hajun)-[:VISITED]->(ramen),
       (seoyeon)-[:VISITED]->(sushi), (seoyeon)-[:VISITED]->(gukbap)
""")
print("맛집 적재 완료. 식당:", len(run_cypher("MATCH (r:Restaurant) RETURN r.name")), "개")

> 오늘의 **따라하기**는 이 맛집 그래프가 아니라 **캠핑장 그래프**로 합니다. 데모에서 본 문법을 **처음 보는 데이터**에 옮겨 보는 것이 따라하기의 목적이라서입니다. 아래 셀이 캠핑장 그래프를 함께 적재합니다(맛집 그래프는 그대로 남습니다). 구조는 나란합니다: **캠핑장**(평점·가격) - **지역** - **시설** - **관리자**(일부 미배정) - **캠퍼**.

따라하기에 쓸 캠핑장 그래프입니다. 맛집과 같은 방식으로 두 장에 나눠 그렸습니다. 캠핑장 6곳을 가운데 두고 관리자 4명·캠퍼 3명이 이어집니다.

<img src="images/캠핑장_그래프_사람.png" width="820">

그리고 캠핑장이 가리키는 **시설과 지역**입니다.

<img src="images/캠핑장_그래프_분류.png" width="820">

In [ ]:
# 따라하기용 캠핑장 시드 적재: 이 셀은 실행만 하세요(맛집 그래프는 그대로 둡니다).
# 아래 따라하기들은 데모가 쓴 맛집이 아니라 이 캠핑장 그래프로 연습합니다.
# 구조는 맛집과 나란합니다: 캠핑장 - 지역 - 시설 - 관리자(일부 미배정) - 캠퍼.
run_cypher("""
CREATE (gangwon:Region {name:'강원'}),
       (jeju:Region {name:'제주'}),
       (gyeonggi:Region {name:'경기'})
CREATE (glamping:Facility {name:'글램핑'}),
       (caravan:Facility {name:'카라반'}),
       (autocamp:Facility {name:'오토캠핑'}),
       (pool:Facility {name:'수영장'})
CREATE (starlight:Campsite {name:'별빛캠핑장', rating:4.8, price:90000}),
       (forest:Campsite {name:'숲속카라반', rating:4.6, price:55000}),
       (wave:Campsite {name:'파도소리', rating:4.2, price:30000}),
       (sunset:Campsite {name:'노을언덕', rating:4.6, price:45000}),
       (starwood:Campsite {name:'별빛숲', rating:4.3, price:38000}),
       (riverside:Campsite {name:'강가마루', rating:4.7, price:70000})
CREATE (starlight)-[:IN_REGION]->(gangwon), (starlight)-[:HAS_FACILITY]->(glamping),
       (forest)-[:IN_REGION]->(gangwon), (forest)-[:HAS_FACILITY]->(caravan),
       (wave)-[:IN_REGION]->(jeju), (wave)-[:HAS_FACILITY]->(autocamp),
       (sunset)-[:IN_REGION]->(jeju), (sunset)-[:HAS_FACILITY]->(pool),
       (starwood)-[:IN_REGION]->(gyeonggi), (starwood)-[:HAS_FACILITY]->(autocamp),
       (riverside)-[:IN_REGION]->(gyeonggi), (riverside)-[:HAS_FACILITY]->(glamping)
CREATE (:Manager {name:'김한별'})-[:MANAGES]->(starlight),
       (:Manager {name:'이나무'})-[:MANAGES]->(forest),
       (:Manager {name:'박노을'})-[:MANAGES]->(sunset),
       (:Manager {name:'정강가'})-[:MANAGES]->(riverside)
CREATE (jiwoo:Camper {name:'지우'}),
       (haram:Camper {name:'하람'}),
       (seoyun:Camper {name:'서윤'})
CREATE (jiwoo)-[:STAYED]->(starlight), (jiwoo)-[:STAYED]->(sunset), (jiwoo)-[:STAYED]->(riverside),
       (haram)-[:STAYED]->(forest), (haram)-[:STAYED]->(starwood),
       (seoyun)-[:STAYED]->(starlight), (seoyun)-[:STAYED]->(wave)
""")
print("캠핑장 적재 완료:", len(run_cypher("MATCH (c:Campsite) RETURN c")), "곳")

> 맛집 그래프입니다. **식당(Restaurant)** 은 `rating`(평점)·`price`(1인 가격) 속성을 갖고, **요리(Cuisine)·지역(Area)·셰프(Chef)** 와 이어집니다. 손님(Diner)은 식당을 `VISITED` 했습니다.

```text
(Diner)-[:VISITED]->(Restaurant)-[:SERVES]->(Cuisine)
                     (Restaurant)-[:LOCATED_IN]->(Area)
                     (Chef)-[:WORKS_AT]->(Restaurant)   ← 셰프 정보가 없는 식당도 있음
```

분석하기 전에 **무엇이 들어 있는지 먼저 훑어봅니다.** 아래 셀은 실행만 하세요.

In [ ]:
# 식당·요리·지역·셰프를 먼저 훑어봅니다(실행만 하세요)
# 두 번째 MATCH 가 앞의 r 을 다시 써서 같은 식당에 조건이 이어진다
for r in run_cypher("MATCH (r:Restaurant)-[:SERVES]->(c:Cuisine) "
                    "MATCH (r)-[:LOCATED_IN]->(a:Area) "
                    "RETURN r.name AS 식당, c.name AS 요리, a.name AS 지역, "
                    "r.rating AS 평점, r.price AS 가격 ORDER BY 식당"):
    print(r['식당'], '/', r['요리'], '/', r['지역'], '/ 평점', r['평점'], '/', r['가격'], '원')

---
# 1. 여러 방식으로 거르기

지난 시간에 `=`·`<>`·`AND`·`OR`·`NOT` 으로 걸러 봤습니다. 여기서는 거르는 **방식 자체를 늘립니다.**

- **1-1** 문자열·목록 연산자로 거르고, 지난 시간처럼 `AND`·`OR` 로 묶습니다.
- **1-2** 정규식과 파라미터 목록을 더합니다.

## 1-1. 목록·부분·접두·접미로 거르기

### 왜 필요할까요?
`=` 하나로는 "이 중 아무거나", "이 글자가 들어간", "이걸로 시작하는" 같은 조건을 표현하기 번거롭습니다. `WHERE` 에 쓰는 **네 가지 연산자**를 더하면 필터가 훨씬 유연해집니다.

### 문법
| 연산자 | 뜻 | 예시 |
|---|---|---|
| `IN [...]` | 목록 중 하나와 일치 | `c.name IN ['일식','양식']` |
| `CONTAINS` | 부분 문자열 포함 | `r.name CONTAINS '홍'` |
| `STARTS WITH` | 접두사로 시작 | `r.name STARTS WITH '스'` |
| `ENDS WITH` | 접미사로 끝 | `r.name ENDS WITH '각'` |

`CONTAINS`·`STARTS WITH`·`ENDS WITH` 는 **문자열 전용**입니다(부분·접두·접미 일치). `IN` 은 목록 안에 값이 있는지 봅니다.

> 앞 시간의 리스트 표현식 `[n IN nodes(p) | n.name]` 에도 `IN` 이 있었지만 **하는 일이 다릅니다.** 거기서는 "목록을 하나씩 훑는다"는 표시였고, 여기서는 "이 값이 목록 안에 있나"를 참·거짓으로 묻는 **비교 연산자**입니다.

In [ ]:
# IN: 일식 또는 양식을 내는 식당
# IN 은 = ... OR = ... 을 목록 하나로 줄여 쓴 것이다
rows = run_cypher("""
MATCH (r:Restaurant)-[:SERVES]->(c:Cuisine)
WHERE c.name IN ['일식', '양식']
RETURN r.name AS 식당, c.name AS 요리
ORDER BY 식당
""")
for r in rows:
    print(r['식당'], '-', r['요리'])

In [ ]:
# CONTAINS / STARTS WITH / ENDS WITH 를 한 번씩 써 본다
# 앞의 셋은 문자열 전용이다. 각각 이름 아무 곳·맨 앞·맨 뒤를 본다
print('이름에 "홍" 포함:',
      [r['n'] for r in run_cypher("MATCH (r:Restaurant) WHERE r.name CONTAINS '홍' RETURN r.name AS n")])
print('이름이 "스"로 시작:',
      [r['n'] for r in run_cypher("MATCH (r:Restaurant) WHERE r.name STARTS WITH '스' RETURN r.name AS n")])
print('이름이 "각"으로 끝:',
      [r['n'] for r in run_cypher("MATCH (r:Restaurant) WHERE r.name ENDS WITH '각' RETURN r.name AS n")])

> 세 연산자는 **대소문자를 가립니다.** 이름이 영문이라면 `STARTS WITH 'S'` 로는 `'sushi'` 가 걸리지 않습니다. 대소문자를 무시하고 찾는 법은 바로 다음 1-2 의 정규식에서 봅니다.

### 새 연산자도 지난 시간처럼 묶습니다

`IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH` 도 결국 `WHERE` 안의 **조건 하나**입니다. 지난 시간에 숫자 비교를 `AND`·`OR` 로 묶었던 것과 똑같이 묶어 씁니다. 새로 배울 문법이 아니라 **새 연산자를 이미 아는 방법으로 조립하는 것**입니다.

In [ ]:
# 같은 두 조건을 AND 로 묶을 때와 OR 로 묶을 때를 나란히 본다
# 두 쿼리는 WHERE 의 낱말 하나만 다르다. 그 한 자리가 결과를 얼마나 바꾸는지 본다
and_rows = run_cypher("""
MATCH (r:Restaurant)-[:SERVES]->(c:Cuisine)
WHERE c.name IN ['일식', '양식'] AND r.price <= 15000
RETURN r.name AS n
ORDER BY n
""")
or_rows = run_cypher("""
MATCH (r:Restaurant)-[:SERVES]->(c:Cuisine)
WHERE c.name IN ['일식', '양식'] OR r.price <= 15000
RETURN r.name AS n
ORDER BY n
""")
# AND 는 둘 다 맞아야 하니 좁아지고, OR 는 하나만 맞아도 되니 넓어진다
print('AND:', [r['n'] for r in and_rows])
print('OR :', [r['n'] for r in or_rows])

> `AND` 는 **1곳**, `OR` 는 **5곳**입니다. 조건 두 개는 그대로인데 이어 붙인 낱말 하나가 질문을 바꿉니다. 조건이 셋 이상 섞이면 **괄호로 우선순위를 못 박는 편**이 안전합니다(`(A OR B) AND C`).

### 🖐️ 함께 따라하기: 접두어와 목록으로 찾기

여기서부터는 **캠핑장 그래프**입니다(데모는 맛집이었죠).

1. 이름이 **`'별'` 로 시작**하는 캠핑장을 `STARTS WITH` 로 찾아 이름을 출력하세요.
2. 시설이 **글램핑 또는 카라반**(`IN`)인 캠핑장도 찾아 이름을 출력하세요.

**확인 기준**: 1번 **2곳** · 2번 **3곳** 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) STARTS WITH '별' 로 캠핑장 이름을 찾아 출력
# 2) HAS_FACILITY 로 시설을 잇고 f.name IN ['글램핑','카라반'] 인 캠핑장 이름을 찾아 출력

### ✅ 바로 확인 퀴즈

**1.** `c.name IN ['일식', '양식']` 과 같은 뜻을 `OR` 로 쓰면?

<details><summary>정답 보기</summary>

`c.name = '일식' OR c.name = '양식'` 입니다. `IN` 은 이 여러 개의 `OR` 을 목록 하나로 짧게 쓴 것입니다.

</details>

**2.** `r.name STARTS WITH '스'`·`r.name ENDS WITH '스'`·`r.name CONTAINS '스'` 는 어떻게 다른가요?

<details><summary>정답 보기</summary>

`STARTS WITH` 는 **맨 앞**, `ENDS WITH` 는 **맨 뒤**가 '스'인 이름만 맞고, `CONTAINS` 는 이름 **아무 곳에나** '스'가 들어가면 맞습니다.

</details>

## 1-2. 정규식과 파라미터 목록

### 왜 필요할까요?
1-1 의 연산자들은 **글자 그대로** 비교합니다. 그런데 실제 질문에는 그것만으로 못 적는 것이 있습니다. "이름이 '효' 또는 '홍' 으로 끝나는 집"처럼 **갈래가 여럿인 조건**, 그리고 대소문자를 가리지 않고 찾아야 하는 영문 이름이 그렇습니다. 또 거를 **목록이 파이썬 쪽에서 정해지는** 경우도 많습니다(사용자가 고른 지역 목록 같은 것). 이 둘을 짧고 안전하게 쓰는 표기가 있습니다.

### 문법 1) 정규식 `=~`
`=~` 는 문자열이 **정규식과 맞는지** 봅니다. 여러 패턴을 `|` 로 나열해 한 줄로 쓸 수 있습니다.

| 쓰는 법 | 뜻 |
|---|---|
| `r.name =~ '스.*'` | '스' 로 시작 |
| `r.name =~ '.*각'` | '각' 으로 끝 |
| `r.name =~ '.*(효\|홍)'` | '효' 또는 '홍' 으로 끝(**갈래 둘을 한 줄로**) |
| `s =~ '(?i)bistro.*'` | **대소문자 무시**하고 'bistro' 로 시작 |

`.` 은 아무 글자 하나, `.*` 은 아무 글자 0개 이상입니다. 그래서 `'.*각'` 은 "앞이 무엇이든 '각' 으로 끝나는" 이 됩니다. `(?i)` 는 맨 앞에 붙이는 **대소문자 무시** 표시입니다(한글에는 대소문자가 없어 영문 데이터에서 씁니다).

> ⚠️ **간단한 조건이면 `STARTS WITH`·`CONTAINS` 를 쓰세요.** 읽기 쉽고, 뒤 단원에서 배울 **인덱스도 그쪽만 활용**합니다. 정규식은 "여러 패턴을 한 번에"처럼 다른 방법이 번거로울 때만 씁니다.

### 문법 2) 목록을 파라미터로: `IN $names`
지난 시간에 배운 `$이름` 자리표시자에 **목록을 통째로** 넘길 수 있습니다. 쿼리 문자열은 그대로 두고 값만 바뀌므로, 목록이 몇 개든 쿼리를 새로 조립할 필요가 없습니다.

```text
WHERE r.name IN $names        ← 쿼리는 고정
run_cypher(쿼리, names=[...])  ← 목록은 파이썬에서 넘긴다
```

In [ ]:
# '.*(효|홍)' 은 '앞이 무엇이든 효 또는 홍 으로 끝나는' 이라는 뜻이다
regex = run_cypher("MATCH (r:Restaurant) WHERE r.name =~ '.*(효|홍)' "
                   "RETURN r.name AS n ORDER BY n")
print('효 또는 홍 으로 끝:', [r['n'] for r in regex])

In [ ]:
# (?i) 대소문자 무시: 한글에는 대소문자가 없으므로 영문 문자열로 확인한다
# MATCH 없이 RETURN 만 쓰면 그래프를 안 보고 식만 계산한다
rows = run_cypher("RETURN 'BistroHong' =~ '(?i)bistro.*' AS 대소문자무시, "
                  "       'BistroHong' =~ 'bistro.*' AS 그냥")
print('(?i) 붙였을 때:', rows[0]['대소문자무시'])
print('안 붙였을 때:', rows[0]['그냥'])

In [ ]:
# 쿼리에는 $names 자리만 두고 값은 파이썬이 넘긴다
# 목록에 없는 이름('없는집')은 그냥 아무것도 맞지 않을 뿐 오류가 아니다
picked = ['스시효', '딤섬각', '없는집']   # 화면에서 사용자가 골랐다고 치자
rows = run_cypher("MATCH (r:Restaurant) WHERE r.name IN $names "
                  "RETURN r.name AS n ORDER BY n", names=picked)
print('고른 이름만 남는다:', [r['n'] for r in rows])

> 목록을 **문자열로 이어 붙이지 마세요.** `"... IN ['" + "','".join(고른것) + "']"` 처럼 쓰면 이름에 따옴표가 섞이는 순간 쿼리가 깨지고, 바깥에서 넣은 값이 쿼리로 실행되는 위험도 생깁니다(**Cypher 주입**). `$names` 로 넘기면 값은 값으로만 다뤄집니다.

### 🖐️ 함께 따라하기: 정규식과 목록으로 캠핑장 고르기

**캠핑장 그래프**입니다.

1. 이름이 **'숲' 또는 '마루' 로 끝나는** 캠핑장을 **정규식 한 줄**로 찾으세요.
2. 파이썬 목록 `['별빛캠핑장', '파도소리', '없는곳']` 을 **파라미터로 넘겨** 그 안에 있는 캠핑장만 찾으세요.

**확인 기준**: 1번 **2곳**(강가마루·별빛숲) · 2번 **2곳** 입니다(목록의 '없는곳' 은 그래프에 없어 그냥 안 나옵니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) WHERE c.name =~ '.*(숲|마루)' 로 캠핑장 이름을 찾아 정렬 출력
# 2) 목록을 변수에 담고 WHERE c.name IN $names 로 넘겨 찾아 정렬 출력

### ✅ 바로 확인 퀴즈

**1.** `r.name =~ '.*각'` 은 어떤 연산자와 같은 뜻인가요?

<details><summary>정답 보기</summary>

**`ENDS WITH '각'`** 과 같습니다. `.*` 이 "앞이 무엇이든"을 뜻하니 "'각' 으로 끝나는 이름"이 됩니다. 갈래가 하나뿐인 이런 조건은 굳이 정규식을 쓸 필요가 없습니다. 정규식은 갈래가 여럿이거나 대소문자를 가리지 않아야 할 때 값을 합니다.

</details>

**2.** 거를 목록이 파이썬 변수에 있을 때, 쿼리 문자열에 이어 붙이지 않고 넘기는 방법은?

<details><summary>정답 보기</summary>

**`WHERE r.name IN $names`** 로 자리표시자를 두고 `run_cypher(쿼리, names=목록)` 으로 넘깁니다. 이어 붙이면 따옴표가 섞일 때 쿼리가 깨지고 **Cypher 주입** 위험도 생깁니다.

</details>

---
# 2. 관계의 있고 없음으로 거르기

1절은 **노드가 가진 값**(이름·평점·가격)으로 걸렀습니다. 그래프에서는 그것 말고도 거를 것이 하나 더 있습니다. **어떤 관계로 무엇과 이어져 있는가** 입니다.

- **2-1** 관계를 이어 붙여 두 단계 이상을 한 번에 묻습니다.
- **2-2** 관계의 존재 자체를 조건으로 씁니다.
- **2-3** 그 조건을 더 넓게 쓰는 표기를 배웁니다.

## 2-1. 관계 여러 종류를 한 패턴에 잇기

### 왜 필요할까요?
지금까지는 한 패턴에 관계가 **한 종류**였습니다. 그런데 실제 질문은 대개 두 단계 이상입니다. "**강남에서 먹은 손님**은 누구인가?" 는 손님이 식당을 방문한 관계(`VISITED`)와 식당이 지역에 속한 관계(`LOCATED_IN`)를 **한 줄로 이어** 물어야 답이 나옵니다.

```text
(Diner)-[:VISITED]->(Restaurant)-[:LOCATED_IN]->(Area)
   손님        방문       식당        위치        지역
```

관계 종류가 달라도 화살표를 계속 이어 붙이면 됩니다. 중간 노드에 조건을 걸 수도 있고, 필요 없으면 변수 이름 없이 `(:Area {name:'강남'})` 처럼 레이블·속성만 적어도 됩니다.

### 이어 붙이기와 헷갈리기 쉬운 것: 관계 종류 중 아무거나
관계를 **이어 붙이는** 것(`-[:A]->(x)-[:B]->`)과, 한 자리에서 **여러 종류 중 아무거나** 허용하는 것(`-[:A|B]->`)은 다릅니다. 세로선 `|` 로 붙이면 "이 관계든 저 관계든"이라는 뜻입니다.

```text
(r)-[:SERVES]->(c)-[:...]->()   한 칸씩 이어 붙이기 (두 단계)
(r)-[:SERVES|LOCATED_IN]->(x)   한 칸인데 관계 종류가 둘 중 아무거나 (한 단계)
```

<img src="images/관계_유형_체인.png" width="760">

*관계 종류가 달라도 화살표를 이어 붙이면 두 단계를 한 번에 물을 수 있습니다.*

In [ ]:
# 관계 두 종류를 한 패턴에: 강남에 있는 식당을 방문한 손님
# 도착 지역은 결과에 안 쓰므로 변수 없이 레이블·속성만 적어 좁힌다
rows = run_cypher("""
MATCH (d:Diner)-[:VISITED]->(r:Restaurant)-[:LOCATED_IN]->(:Area {name:'강남'})
RETURN d.name AS 손님, r.name AS 식당
ORDER BY 손님, 식당
""")
for row in rows:
    print(row['손님'], '-', row['식당'])

> 한 줄에 `VISITED` 와 `LOCATED_IN` 을 이어 붙였습니다. 관계를 하나씩 따로 물었다면 두 번 조회해 파이썬에서 맞춰야 했을 텐데, 그래프에서는 **패턴 하나**로 끝납니다.

In [ ]:
# 관계 종류 중 아무거나로 한 칸에 닿는 곳
# 세로선(|)은 한 칸짜리 관계의 종류를 둘 중 아무거나로 여는 표기다.
# type(x) 로 그 행이 어느 관계를 탄 것인지 되짚는다
rows = run_cypher("""
MATCH (r:Restaurant {name:'스시효'})-[x:SERVES|LOCATED_IN]->(t)
RETURN type(x) AS 관계종류, t.name AS 닿는곳
ORDER BY 닿는곳
""")
for row in rows:
    print(row['관계종류'], '->', row['닿는곳'])

> 한 번의 조회로 지역(강남)과 요리(일식)가 **두 줄로** 함께 나옵니다. `type(x)` 는 그 행이 어떤 관계를 탄 것인지 알려 줍니다. "이 노드에서 나가는 관계를 종류 상관없이 훑고 싶다"면 이 표기가 편합니다.

### 🖐️ 함께 따라하기: 세 종류의 관계 잇기

다시 **캠핑장 그래프**입니다. 관계 **세 종류**를 이어 봅니다. **캠퍼 → 캠핑장 → 시설**(`STAYED`·`HAS_FACILITY`)을 한 패턴으로 잇고, 거기에 **관리자 → 캠핑장**(`MANAGES`) 패턴을 한 줄 더(`MATCH` 를 한 번 더 써서) 붙이세요. 캠퍼·캠핑장·시설·관리자 네 값을 캠퍼 이름, 캠핑장 이름 순으로 출력합니다.

**확인 기준**: **5행**이 나옵니다. 관리자가 없는 캠핑장은 이 쿼리에서 **빠집니다**. 왜 그런지는 3-1 의 `OPTIONAL MATCH` 에서 다룹니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) (cp:Camper)-[:STAYED]->(c:Campsite)-[:HAS_FACILITY]->(f:Facility) 패턴을 적는다
# 2) MATCH 를 한 줄 더 써서 (m:Manager)-[:MANAGES]->(c) 를 잇는다 (같은 c 를 다시 쓴다)
# 3) 캠퍼·캠핑장·시설·관리자를 RETURN 하고 캠퍼·캠핑장 순으로 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `(d:Diner)-[:VISITED]->(r:Restaurant)-[:LOCATED_IN]->(a:Area)` 는 무엇을 찾나요?

<details><summary>정답 보기</summary>

손님이 방문한 식당과, **그 식당이 속한 지역**까지 한 번에 찾습니다. 관계 종류가 달라도 화살표를 이어 붙이면 두 단계를 한 패턴으로 물을 수 있습니다.

</details>

**2.** 중간 노드의 이름이 결과에 필요 없다면 어떻게 쓸 수 있나요?

<details><summary>정답 보기</summary>

변수 없이 레이블·속성만 적습니다. `(:Area {name:'강남'})` 처럼요. 변수를 안 붙이면 뒤에서 그 노드를 쓸 수 없지만, 패턴을 좁히는 역할은 그대로 합니다.

</details>

## 2-2. 패턴 자체를 조건으로 쓰기

### 왜 필요할까요?
"**셰프가 있는** 식당만", "**아직 아무도 안 간** 식당만", "**강남에 있는** 곳만". 이런 질문의 조건은 노드의 값이 아니라 **관계가 있느냐 없느냐** 입니다.

2-1 처럼 패턴을 `MATCH` 에 이어 붙여도 "있는 것"은 찾을 수 있습니다. 그런데 그렇게 하면 **셰프가 결과에 딸려 나오고**, 식당 하나에 셰프가 둘이면 **행이 두 줄로 늘어납니다.** 우리가 원한 건 식당 목록뿐인데 말입니다. 그리고 "**없는 것**"은 이 방법으로 아예 찾을 수 없습니다. 패턴이 안 맞는 행은 지워지니까요.

### 문법: `WHERE` 에 패턴을 그대로 적는다
`WHERE` 자리에 **패턴을 적으면** "그런 관계가 하나라도 있는가"를 참·거짓으로 봅니다. 이렇게 조건 자리에 적은 패턴을 **패턴 술어**라고 부릅니다(뒤에서 이 이름으로 다시 나옵니다).

| 쓰는 법 | 뜻 |
|---|---|
| `WHERE (r)<-[:WORKS_AT]-(:Chef)` | 셰프가 **있는** 식당만 |
| `WHERE NOT (r)<-[:WORKS_AT]-(:Chef)` | 셰프가 **없는** 식당만 |
| `WHERE (r)-[:LOCATED_IN]->(:Area {name:'강남'})` | 강남에 있는 식당만 |

**규칙 두 가지**만 지키면 됩니다.

1. **이미 있는 변수**(`r`)를 한쪽 끝에 두고 이어 씁니다. 여기서 **새 변수를 만들 수는 없습니다**(`(r)<-[:WORKS_AT]-(ch:Chef)` 처럼 이름을 붙이면 안 됩니다). 이름이 필요 없으니 `(:Chef)` 처럼 레이블만 적습니다.
2. 관계가 **하나 이상** 있어야 합니다. `WHERE (r)` 처럼 노드만 적을 수는 없습니다.

행이 늘지 않고, 딸려 나오는 값도 없고, `NOT` 하나로 "없는 것"까지 뒤집을 수 있습니다.

<img src="images/관계_있고_없음.png" width="760">

*같은 식당 목록을 두 갈래로 가릅니다. 셰프로 이어지는 화살표가 있는 쪽과 없는 쪽입니다.*

In [ ]:
# 두 쿼리는 NOT 한 단어만 다르다
# 패턴 술어는 '이런 관계가 하나라도 있나' 를 참·거짓으로 본다.
# 패턴 술어 안에는 새 변수를 못 만든다. 레이블만 적는다
with_chef = run_cypher("""
MATCH (r:Restaurant)
WHERE (r)<-[:WORKS_AT]-(:Chef)
RETURN r.name AS 식당 ORDER BY 식당
""")
# NOT 은 그 패턴이 '하나도 없는' 것만 남긴다
without_chef = run_cypher("""
MATCH (r:Restaurant)
WHERE NOT (r)<-[:WORKS_AT]-(:Chef)
RETURN r.name AS 식당 ORDER BY 식당
""")
print('셰프가 있는 식당:', [r['식당'] for r in with_chef])
print('셰프가 없는 식당:', [r['식당'] for r in without_chef])

> 여섯 곳이 4곳과 2곳으로 정확히 갈렸습니다. **셰프 이름은 결과에 하나도 나오지 않습니다.** 우리가 물은 것이 식당 목록이었으니까요. `MATCH` 로 이어 붙였다면 셰프 열이 딸려 나오고, 셰프가 없는 두 곳은 아예 사라졌을 것입니다.

In [ ]:
# 패턴 조건도 다른 조건처럼 AND / OR 로 묶인다
# 여기서는 '셰프가 있고' 와 '5만 원 이하' 를 함께 만족하는 곳만 남긴다
rows = run_cypher("""
MATCH (r:Restaurant)
WHERE (r)<-[:WORKS_AT]-(:Chef) AND r.price <= 50000
RETURN r.name AS 식당, r.price AS 가격
ORDER BY 가격
""")
for r in rows:
    print(r['식당'], r['가격'])

### 🖐️ 함께 따라하기: 시설이 있는 곳과 없는 곳

**캠핑장 그래프**입니다. 시설(`HAS_FACILITY`)로 두 번 물어 보세요.

1. **글램핑 시설이 있는** 캠핑장만 찾으세요.
2. **오토캠핑 시설이 없는** 캠핑장만 찾으세요(`NOT` 을 붙입니다).

**확인 기준**: 1번 **2곳**(강가마루·별빛캠핑장) · 2번 **4곳** 입니다. 시설 이름은 결과에 나오지 않아야 합니다(캠핑장 이름만).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Campsite) 로 캠핑장을 잡고, WHERE 에 (c)-[:HAS_FACILITY]->(:Facility {name:'글램핑'}) 를 적는다
# 2) 같은 쿼리에서 시설 이름을 오토캠핑으로 바꾸고 앞에 NOT 을 붙인다
# 3) 두 결과를 캠핑장 이름만 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `MATCH (r:Restaurant)-[:LOCATED_IN]->(:Area {name:'강남'})` 와 `MATCH (r:Restaurant) WHERE (r)-[:LOCATED_IN]->(:Area {name:'강남'})` 는 결과가 같은가요?

<details><summary>정답 보기</summary>

이 데이터에서는 **같습니다**(식당 하나가 지역 하나에만 속하니까요). 하지만 한 식당이 지역 여러 곳에 이어져 있다면 위쪽은 **행이 그만큼 늘어나고** 아래쪽은 식당당 한 줄입니다. "조건으로만 쓸" 때는 `WHERE` 쪽이 안전합니다.

</details>

**2.** `WHERE NOT (r)<-[:WORKS_AT]-(ch:Chef)` 라고 쓰면 어떻게 되나요?

<details><summary>정답 보기</summary>

**오류가 납니다.** `WHERE` 에 적는 패턴은 **새 변수를 만들 수 없습니다.** 셰프 이름이 필요 없으니 `(:Chef)` 처럼 변수 없이 적어야 합니다.

</details>

## 2-3. `EXISTS { }` 로 넓혀 쓰기

### 왜 필요할까요?
2-2 의 표기는 짧지만 **새 변수를 못 만든다**는 제약이 있습니다. 그래서 "**이름이 '지' 로 시작하는** 손님이 다녀간 식당" 처럼 **상대 쪽에 조건을 걸어야** 하는 질문은 쓸 수 없습니다. `(:Diner {name:'지민'})` 처럼 값이 딱 정해진 것은 속성 map 으로 적을 수 있지만, `STARTS WITH` 같은 조건은 map 에 못 넣기 때문입니다.

### 문법: `EXISTS { 패턴 }`
중괄호 안에 **작은 조회 하나**를 통째로 넣습니다. 그 안에서는 **변수를 만들 수 있고 `WHERE` 도 쓸 수 있습니다.** 맞는 것이 하나라도 있으면 참입니다.

```text
WHERE EXISTS { (r)<-[:VISITED]-(d:Diner) WHERE d.name STARTS WITH '지' }
               └ 바깥 변수 r 로 시작   └ 여기선 새 변수 d 를 만들어도 된다
```

| 쓰는 법 | 되나 | 언제 쓰나 |
|---|---|---|
| `WHERE (r)<-[:WORKS_AT]-(:Chef)` | O | 짧다. 관계 유무만 볼 때 |
| `WHERE EXISTS { (r)<-[:WORKS_AT]-(:Chef) }` | O | 위와 같은 뜻. 길지만 뜻이 또렷하다 |
| `WHERE (r)<-[:VISITED]-(d) WHERE d.name STARTS WITH '지'` | X | 패턴 술어에는 `WHERE` 를 못 쓴다 |
| `WHERE EXISTS { (r)<-[:VISITED]-(d:Diner) WHERE d.name STARTS WITH '지' }` | O | 이럴 때 쓰는 표기 |

**둘 중 무엇을 쓸까요.** 관계가 있는지만 보면 2-2 의 짧은 표기, 상대 쪽에 조건이 붙으면 `EXISTS { }` 입니다. `NOT` 은 양쪽 다 앞에 붙일 수 있습니다(`NOT EXISTS { ... }`).

In [ ]:
# 같은 질문을 짧은 패턴 술어와 EXISTS 표기로 각각 물어본다
# 관계 유무만 보는 질문이라 둘의 답은 같다. 표기만 다르다
pattern_form = run_cypher("MATCH (r:Restaurant) WHERE (r)<-[:WORKS_AT]-(:Chef) "
                  "RETURN r.name AS n ORDER BY n")
exists_form = run_cypher("MATCH (r:Restaurant) WHERE EXISTS { (r)<-[:WORKS_AT]-(:Chef) } "
                        "RETURN r.name AS n ORDER BY n")
print('패턴 술어 :', [r['n'] for r in pattern_form])
print('EXISTS   :', [r['n'] for r in exists_form])

In [ ]:
# EXISTS 여야만 되는 자리: 상대 쪽(손님)에 조건을 건다
# 중괄호 안에서는 새 변수 d 를 만들고 WHERE 도 걸 수 있다
rows = run_cypher("""
MATCH (r:Restaurant)
WHERE EXISTS { (r)<-[:VISITED]-(d:Diner) WHERE d.name STARTS WITH '지' }
RETURN r.name AS 식당
ORDER BY 식당
""")
print("'지' 로 시작하는 손님이 다녀간 식당:", [r['식당'] for r in rows])

> 손님 중 '지' 로 시작하는 이름은 지민 한 명이고, 그가 다녀간 식당 3곳이 나왔습니다. **손님 이름은 결과에 나오지 않습니다.** `EXISTS { }` 안에서 만든 변수는 중괄호 밖으로 나오지 못하기 때문입니다. 그래서 "조건으로만 쓰고 결과에는 넣지 않는다"는 뜻이 문법으로 드러납니다.

### 🖐️ 함께 따라하기: 특정 캠퍼가 묵은 캠핑장

**캠핑장 그래프**에서 **이름이 `'우'` 로 끝나는 캠퍼**가 묵은(`STAYED`) 캠핑장을 찾으세요. 캠퍼 쪽에 조건이 붙으므로 `EXISTS { }` 안에서 캠퍼 변수를 만들어야 합니다.

**확인 기준**: **3곳**(강가마루·노을언덕·별빛캠핑장) 이 나오고, 캠퍼 이름은 결과에 나오지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Campsite) 로 캠핑장을 잡는다
# 2) WHERE EXISTS { (c)<-[:STAYED]-(cp:Camper) WHERE cp.name ENDS WITH '우' } 를 적는다
# 3) 캠핑장 이름만 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** 2-2 의 짧은 표기 대신 `EXISTS { }` 를 써야 하는 때는 언제인가요?

<details><summary>정답 보기</summary>

**상대 쪽 노드에 조건을 걸어야 할 때**입니다. 짧은 표기는 새 변수를 만들 수 없어서 `WHERE d.name STARTS WITH '지'` 같은 조건을 붙일 자리가 없습니다.

</details>

**2.** `EXISTS { }` 안에서 만든 변수를 `RETURN` 에 쓸 수 있나요?

<details><summary>정답 보기</summary>

**쓸 수 없습니다.** 중괄호 밖으로 나오지 못합니다. 그 값을 결과에 넣고 싶다면 `EXISTS` 가 아니라 `MATCH` 로 이어 붙이거나, 다음 절의 `OPTIONAL MATCH` 를 써야 합니다.

</details>

---
# 3. `OPTIONAL MATCH`: 짝이 없어도 빠뜨리지 않기

2절에서는 관계가 있는 것만, 또는 없는 것만 **골랐습니다.** 여기서는 **양쪽을 한 표에 담습니다.**

- **3-1** 짝이 없어도 행을 남깁니다.
- **3-2** 그 결과를 거를 때 빠지는 함정을 봅니다.
- **3-3** 이 성질로 두 노드가 닿는지 판별하고, 화살표가 질문의 뜻을 바꾸는 것까지 봅니다.

## 3-1. 짝이 없어도 행을 남기기

### 왜 필요할까요?
"모든 식당과 그 셰프"를 **한 표로** 뽑고 싶은데, **셰프 정보가 없는 식당**(국밥천국·라멘야마)이 있습니다. 보통 `MATCH` 로 셰프를 이으면, 셰프가 없는 식당은 **아예 결과에서 사라집니다**. 그러면 그 표는 "식당 목록"이 아니게 됩니다.

> 2-2 에서 "셰프가 없는 식당"만 따로 뽑았던 것과는 다른 질문입니다. 거기서는 **한쪽만** 골랐고, 여기서는 **여섯 곳을 다 남기되 셰프 칸을 비워** 둡니다.

### 문법: `OPTIONAL MATCH`
`OPTIONAL MATCH` 는 패턴이 맞으면 이어 주고, **안 맞아도 행을 지우지 않고** 그 자리를 `null` 로 둡니다 (관계형 DB의 LEFT JOIN 과 같은 생각).

```text
MATCH (r:Restaurant)                       ← 모든 식당은 반드시 남는다
OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r)  ← 셰프가 있으면 잇고, 없으면 ch 는 null
```

<img src="images/optional_match_null.png" width="760">

*같은 질문을 `MATCH` 로 물으면 4행, `OPTIONAL MATCH` 로 물으면 6행: 셰프가 없는 두 식당이 살아남고 셰프 자리만 `null` 이 됩니다.*

In [ ]:
# 셰프 없는 식당은 셰프 자리가 None 으로 남는다
# 첫 MATCH 가 남길 행을 정하고, OPTIONAL MATCH 는 붙일 수 있으면 붙인다.
# 그냥 MATCH 로 이었다면 셰프 없는 두 곳이 결과에서 사라져 4행이 된다
rows = run_cypher("""
MATCH (r:Restaurant)
OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r)
RETURN r.name AS 식당, ch.name AS 셰프
ORDER BY 식당
""")
for r in rows:
    print(r['식당'], '→', r['셰프'])

> 국밥천국·라멘야마는 셰프가 `None`(파이썬에서 본 `null`)으로 나옵니다. `MATCH` 였다면 이 두 식당은 사라졌을 텐데, `OPTIONAL MATCH` 라서 남았습니다.

### 🖐️ 함께 따라하기: 모든 캠핑장과 (있으면) 관리자

**캠핑장 그래프**에서 **모든 캠핑장**을 남기고 관리자(`MANAGES`)를 있으면 이어 한 표로 출력하세요. 관리자가 없는 곳은 그 칸이 `None` 으로 나와야 합니다.

이어서 같은 쿼리에서 `OPTIONAL MATCH` 를 그냥 `MATCH` 로 바꿔 다시 실행하고, **행 수가 어떻게 달라지는지** 비교하세요.

**확인 기준**: `OPTIONAL MATCH` 는 **6행**, `MATCH` 는 **4행** 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Campsite) 로 캠핑장을 모두 잡고 OPTIONAL MATCH 로 (m:Manager)-[:MANAGES]->(c) 를 잇는다
# 2) 캠핑장 이름과 관리자 이름을 RETURN 해 정렬 출력한다 (관리자 없는 곳은 None)
# 3) OPTIONAL 을 뺀 같은 쿼리를 한 번 더 실행해 행 수를 비교한다

### ✅ 바로 확인 퀴즈

**1.** 그냥 `MATCH (ch:Chef)-[:WORKS_AT]->(r)` 로 이으면 셰프 없는 식당은 어떻게 되나요?

<details><summary>정답 보기</summary>

결과에서 **사라집니다**. 패턴이 맞지 않는 행은 지워지기 때문입니다. 그래서 셰프 없는 식당을 남기려면 `OPTIONAL MATCH` 를 써야 합니다.

</details>

**2.** `OPTIONAL MATCH` 로 이었을 때 짝이 없으면 그 자리에 무엇이 들어가나요?

<details><summary>정답 보기</summary>

**`null`**(파이썬에서는 `None`)이 들어갑니다. 행은 남고 값만 비어 있습니다.

</details>

## 3-2. `WHERE` 를 어디에 거느냐

### 왜 필요할까요?
3-1 의 표에서 **셰프가 없는 두 곳만** 남기고 싶다고 해 봅시다. `OPTIONAL MATCH` 줄 바로 뒤에 `WHERE ch IS NULL` 을 붙이고 싶어집니다. 자연스러워 보이는데, **그렇게 하면 하나도 걸러지지 않습니다.** 먼저 실행해 눈으로 보고 그 이유를 봅시다.

In [ ]:
# 함정: OPTIONAL MATCH 바로 뒤의 WHERE 는 걸러 주지 않는다
# 이 자리의 WHERE 는 '거르는 조건' 이 아니라 '찾을 패턴의 조건' 이다.
# 그래서 하나도 못 찾고 식당마다 ch 가 null 인 행만 남는다
rows = run_cypher("""
MATCH (r:Restaurant)
OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r)
WHERE ch IS NULL
RETURN r.name AS 식당
ORDER BY 식당
""")
print('걸러지지 않고 다 나온다:', [r['식당'] for r in rows])

> 걸러지지 않았습니다. `OPTIONAL MATCH` 바로 뒤의 `WHERE` 는 **그 패턴을 찾을 때 쓰는 조건**입니다. 조건에 안 맞으면 행이 지워지는 것이 아니라 **짝을 못 찾은 것으로 쳐서 그 자리가 `null` 이 됩니다.** 그냥 `MATCH` 였다면 안 맞는 행이 통째로 지워져 걸러졌겠지만, `OPTIONAL MATCH` 는 행을 남기는 것이 일이라 그렇게 되지 않습니다.
>
> 이미 나온 결과를 거르려면 **`WITH` 로 한 단계 넘긴 뒤** `WHERE` 를 겁니다.

```text
MATCH (r:Restaurant)
OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r)
WITH r, ch          ← 여기까지 나온 결과를 다음 단계로 넘긴다
WHERE ch IS NULL    ← 넘어온 결과를 거른다
```

`WITH` 는 4절에서 자세히 다룹니다. 지금은 **"앞 단계 결과를 다음 단계로 넘긴다"** 는 뜻만 알면 됩니다. 바로 아래에서 같은 조회에 이 한 줄을 끼워 실행해 봅니다.

In [ ]:
# 앞 셀과 WITH r, ch 한 줄만 다르다. 이번에는 걸러진다
# WITH 가 '여기까지가 한 단계' 를 그어 주므로,
# 그다음 줄의 WHERE 는 찾을 조건이 아니라 넘어온 결과를 거르는 조건이 된다
rows = run_cypher("""
MATCH (r:Restaurant)
OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r)
WITH r, ch
WHERE ch IS NULL
RETURN r.name AS 식당
ORDER BY 식당
""")
print('셰프가 없는 식당:', [r['식당'] for r in rows])

> 이번에는 2곳으로 걸러졌습니다. 쿼리에서 달라진 것은 **`WITH r, ch` 한 줄**뿐입니다. 그 줄이 단계를 끊어 주니, 똑같은 `WHERE ch IS NULL` 이 "찾을 조건" 에서 "거를 조건" 으로 바뀐 것입니다.

> 참고로 "셰프가 없는 식당" **만** 알고 싶었다면 2-2 의 `WHERE NOT (r)<-[:WORKS_AT]-(:Chef)` 가 훨씬 짧습니다. `OPTIONAL MATCH` + `WITH` + `WHERE` 는 **짝을 붙인 표를 만든 뒤** 그중 일부를 걸러야 할 때 쓰는 방법입니다.

### 🖐️ 함께 따라하기: 관리자가 없는 캠핑장만

**캠핑장 그래프**에서 같은 함정을 피해 봅니다. **관리자가 등록되지 않은 캠핑장**만 골라 이름을 출력하세요. `OPTIONAL MATCH` 다음 줄에서 `WITH c, m` 으로 한 단계 넘기고, 그다음 줄에 `WHERE m IS NULL` 을 겁니다.

**확인 기준**: `['별빛숲', '파도소리']` 두 곳이 나옵니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 모든 캠핑장 -> OPTIONAL MATCH 로 관리자를 (있으면) 잇는다
# 2) WITH c, m 으로 한 단계 넘긴 뒤, 그다음 줄 WHERE m IS NULL 로 관리자 없는 캠핑장만 남긴다
# 3) 캠핑장 이름만 리스트로 출력한다

### ✅ 바로 확인 퀴즈

**1.** `OPTIONAL MATCH` 로 이은 결과에서 짝이 없는 행만 남기려면 `WHERE ch IS NULL` 을 어디에 적나요?

<details><summary>정답 보기</summary>

**`WITH` 로 한 단계 넘긴 뒤**에 적습니다. `OPTIONAL MATCH` 줄 바로 뒤에 붙인 `WHERE` 는 그 패턴을 찾을 때 쓰는 조건이라 이미 나온 결과를 거르지 못합니다(식당 6곳이 그대로 나옵니다).

</details>

**2.** "셰프가 없는 식당 이름만" 이 필요하다면 2-2 와 3-2 중 어느 쪽이 낫나요?

<details><summary>정답 보기</summary>

**2-2** 입니다. `WHERE NOT (r)<-[:WORKS_AT]-(:Chef)` 한 줄이면 끝납니다. 3-2 의 방법은 `OPTIONAL MATCH` 로 만든 **표에서** 일부를 걸러야 할 때(짝을 함께 봐야 할 때) 씁니다.

</details>

## 3-3. 두 노드가 닿는지 판별하기

### 왜 필요할까요?
`OPTIONAL MATCH` 의 "있으면 담고, 없으면 `null`" 성질은 **두 노드가 서로 닿는지**를 참·거짓으로 딱 잘라 판별하는 데도 씁니다. 앞 시간의 `shortestPath` 로 경로를 `p` 에 담되 **`OPTIONAL MATCH` 로 감싸면**, 경로가 없을 때 빈 결과가 아니라 `p = null` 이 됩니다. 그 뒤 **`p IS NOT NULL`** 을 물으면 연결 여부가 `True`/`False` 로 나옵니다.

> 그냥 `MATCH` 로 쓰면 경로가 없을 때 **행 자체가 안 옵니다.** 그러면 파이썬에서 `rows[0]` 이 `IndexError` 를 냅니다. `OPTIONAL MATCH` 로 감싸면 **항상 한 행**이 오므로 판별 함수를 만들기 쉽습니다.

### 무엇으로 이어졌다고 볼 것인가
두 손님 사이에는 직접 이어진 관계가 없습니다. 대신 **같은 식당을 방문했다면 그 식당을 다리 삼아** `VISITED` 관계로 이어집니다. 아래 판별은 그 이어짐이 있는지를 봅니다. 관계를 `VISITED` 하나로 좁혔으니, 겹치는 식당이 없는 두 손님은 `False` 가 됩니다.

In [ ]:
# 경로가 없으면 OPTIONAL MATCH 라 p 가 null 이 되어 connected 가 False
def connected(name1, name2):
    """두 손님이 VISITED 관계를 타고 서로 닿는지 True/False 로 돌려준다."""
    # 첫 MATCH 로 두 손님을 잡고, 경로 찾기만 OPTIONAL 로 감싼다.
    # shortestPath 로 감싸 첫 답에서 멈추게 한다
    rows = run_cypher("""
    MATCH (a:Diner {name:$n1}), (b:Diner {name:$n2})
    OPTIONAL MATCH p = shortestPath( (a)-[:VISITED*]-(b) )
    RETURN p IS NOT NULL AS connected
    """, n1=name1, n2=name2)
    # 경로가 없어도 행은 남으므로 rows 는 항상 한 행이다
    return rows[0]['connected']


print('지민 ↔ 서연:', connected('지민', '서연'))
print('지민 ↔ 하준:', connected('지민', '하준'))

> 지민·서연은 **스시효**를 둘 다 방문해 `VISITED` 로 이어지므로 `True`, 지민·하준은 겹치는 식당이 없어 `False` 입니다. 경로가 없어도 `OPTIONAL MATCH` 라 행이 남고 `p` 가 `null` 이 되어, `p IS NOT NULL` 이 `False` 로 깔끔히 판별됩니다.

> ⚠️ `-[:VISITED*]-` 처럼 **상한이 없는 가변길이**는 길이 1, 2, 3, ... 으로 넓혀 가며 훑어서, 큰 그래프에서는 훑는 경로가 폭발합니다. `*1..4` 처럼 상한을 두거나 `shortestPath` 로 감싸 첫 답에서 멈추게 하세요. 위에서 감싼 이유가 그것입니다.

### 🖐️ 함께 따라하기: 두 캠퍼가 이어지는가

**캠핑장 그래프**에서 두 캠퍼가 **같은 캠핑장에 묵은 것을 다리 삼아**(`STAYED`) 이어지는지 판별하는 함수 `is_connected(a, b)` 를 만드세요. 이름은 파라미터로 넘깁니다.

**확인 기준**: `is_connected('지우', '서윤')` 는 **True**(별빛캠핑장에 둘 다 묵었습니다), `is_connected('지우', '하람')` 은 **False** 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) is_connected(a, b) 함수를 정의한다
# 2) MATCH 로 두 Camper 를 잡고, OPTIONAL MATCH p = shortestPath((x)-[:STAYED*]-(y)) 로 감싼다
# 3) RETURN p IS NOT NULL 을 받아 rows[0] 의 값을 돌려준다
# 4) ('지우','서윤') 과 ('지우','하람') 으로 각각 호출해 출력한다

### 화살표 하나가 질문을 바꾼다

위 판별에서 관계를 **`-[:VISITED*]-`**(화살표 없이) 로 썼습니다. 그런데 `VISITED` 는 원래 **손님 → 식당** 방향의 관계입니다. 화살표를 살려 `-[:VISITED*]->` 로 쓰면 "손님에서 출발해 계속 앞으로만 가서 다른 손님에 닿을 수 있나"를 묻게 되는데, 식당에서 손님으로 나가는 화살표가 없으니 **영영 닿지 못합니다**.

같은 두 손님, 같은 질문인데 **화살표 하나로 답이 갈립니다**. 아래 한 셀에서 직접 확인하세요.

<img src="images/경로_방향.png" width="760">

*화살표를 빼면 식당에서 손님 쪽으로도 거슬러 갈 수 있는 셈이라, 같은 두 손님의 답이 뒤집힙니다.*

In [ ]:
# 같은 두 손님을 화살표 없이 / 화살표를 지켜서 물어본다
# 경로를 p1·p2 에 따로 담아 한 행에서 두 답을 받는다.
# VISITED 는 손님 -> 식당 방향이라 되나올 길이 없다
rows = run_cypher("""
MATCH (a:Diner {name:'지민'}), (b:Diner {name:'서연'})
OPTIONAL MATCH p1 = shortestPath( (a)-[:VISITED*]-(b) )
OPTIONAL MATCH p2 = shortestPath( (a)-[:VISITED*]->(b) )
RETURN p1 IS NOT NULL AS 방향무시, p2 IS NOT NULL AS 방향준수
""")
print('화살표 없이 -[:VISITED*]- :', rows[0]['방향무시'])
print('화살표 지켜 -[:VISITED*]-> :', rows[0]['방향준수'])

> 방향을 무시하면 `True`, 화살표를 지키면 `False` 입니다. 관계에 방향이 있는 데이터에서는 **화살표를 쓸지 말지가 곧 질문의 뜻**입니다. 지하철 인접(`NEXT_TO`)처럼 양쪽으로 오갈 수 있는 관계는 화살표 없이, 배송 노선·선수과목처럼 한쪽으로만 흐르는 관계는 화살표를 지켜 물어야 답이 맞습니다.

### 🖐️ 함께 따라하기: 화살표 하나로 결과가 달라지는지 직접 보기

이번엔 **캠핑장과 지역**으로 확인합니다. `IN_REGION` 은 **캠핑장 → 지역** 방향의 관계입니다.

`별빛캠핑장` 에서 `IN_REGION` 을 **두 칸까지**(`*1..2`) 따라가 닿는 곳을 두 번 구하세요.

1. 화살표를 **지켜서** `-[:IN_REGION*1..2]->`
2. 화살표 **없이** `-[:IN_REGION*1..2]-`

**확인 기준**: 1번은 **1곳**(강원)뿐이고, 2번은 **2곳**입니다. 화살표를 빼면 지역에서 다시 거꾸로 내려가 **같은 지역의 다른 캠핑장**까지 잡히기 때문입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 별빛캠핑장에서 IN_REGION 을 화살표 지켜(->) *1..2 로 따라가 닿는 곳 이름을 DISTINCT 로 출력한다
# 2) 같은 쿼리에서 화살표만 빼고(-) 다시 실행해 출력한다
# 3) 두 결과의 개수를 비교한다 (1곳 대 2곳)

### ✅ 바로 확인 퀴즈

**1.** `MATCH p = shortestPath(...)` 로 쓰면 경로가 없을 때 어떻게 되나요?

<details><summary>정답 보기</summary>

**행이 하나도 오지 않습니다.** 그래서 `rows[0]` 이 `IndexError` 를 냅니다. `OPTIONAL MATCH` 로 감싸면 행은 오고 `p` 만 `null` 이 되어 판별이 깔끔해집니다.

</details>

**2.** `-[:R*]-` 와 `-[:R*]->` 는 같은 질문인가요? 배송 노선처럼 한쪽으로만 흐르는 관계를 화살표 없이 물으면 어떤 문제가 생기나요?

<details><summary>정답 보기</summary>

**아닙니다.** 화살표가 없으면 관계를 **어느 쪽으로든** 타고 갈 수 있고, 화살표가 있으면 **그 방향으로만** 갑니다. 배송 노선을 화살표 없이 물으면 노선을 **거꾸로 거슬러** 올라가는 경로까지 답으로 잡혀, 실제로는 갈 수 없는 길인데 "갈 수 있다"고 나옵니다.

</details>

**3.** 그렇다면 지하철 인접(`NEXT_TO`)이나 SNS 친구(`FRIEND`) 는 왜 화살표를 빼고 물어도 되나요?

<details><summary>정답 보기</summary>

그 관계들은 **원래 방향이 없는 사이**이기 때문입니다. 데이터로 만들 때는 한쪽 방향으로 한 번만 저장하지만 (A-B 와 B-A 를 둘 다 만들면 중복입니다), 뜻은 양쪽이 같습니다. 그래서 조회할 때 화살표를 빼야 저장한 방향과 상관없이 제대로 찾힙니다. **저장 방향이 뜻을 갖느냐**가 기준입니다.

</details>

---
# 4. `WITH` 파이프라인: 단계별로 다듬기

복잡한 쿼리는 한 번에 쓰기 어렵습니다. **단계별로** 좁혀 가면 훨씬 쉽습니다.

- **4-1** 값을 그대로 넘깁니다.
- **4-2** 값을 새로 만들어 넘깁니다.
- **4-3** 중간에서 잘라 다음 단계로 넘깁니다.

> 1-1 에서 쓴 `STARTS WITH`·`ENDS WITH` 의 `WITH` 와는 **아무 관계가 없습니다.** 그건 연산자 이름의 일부이고, 여기서 배우는 `WITH` 는 중간 결과를 다음 절로 넘기는 **절**입니다. 이름이 겹칠 뿐입니다.

## 4-1. 값을 넘기고 한 번 더 거르기

### 왜 필요할까요?
"평점 4.5 이상만 남기고 → 그중 5만 원 이하만 → 평점 높은 순"처럼 **단계별로** 좁혀 가면 복잡한 질문도 한 줄씩 쌓아 만들 수 있습니다. `WITH` 는 한 단계의 결과를 **다음 단계로 넘기는 파이프**입니다.

### 문법: `WITH` 로 넘기기
`WITH` 는 `RETURN` 처럼 값을 고르지만, **끝내지 않고 다음 절로 넘깁니다**. 넘긴 것에 다시 `WHERE` 를 걸거나 `DISTINCT` 로 중복을 없앨 수 있습니다. 지난 시간의 `RETURN DISTINCT` 가 **결과를 돌려주면서** 중복을 없앴다면, `WITH DISTINCT` 는 **중간에서** 없애고 그다음 절을 이어 갑니다.

- **값 전달**: `WITH r`: 앞에서 찾은 `r` 을 다음 단계로 넘김
- **단계 필터**: `WITH r WHERE r.price <= 30000`: 넘기면서 한 번 더 거름
- **중복 제거**: `WITH DISTINCT c.name AS 요리`: 서로 다른 값만 남김

### 언제 `AND` 로 충분하고, 언제 `WITH` 가 필요한가
아래 데모의 두 조건(`평점 4.5 이상`·`가격 5만 원 이하`)은 사실 **`AND` 한 줄로도 됩니다.** 조건을 나란히 거는 것뿐이라면 `AND` 가 짧고 빠릅니다. `WITH` 가 꼭 필요한 때는 **중간에 결과를 손봐야 할 때**입니다.

- 중복을 없앤 뒤에 다시 걸러야 할 때(`WITH DISTINCT ...`)
- 중간에 정렬해 몇 개만 남기고 다음 단계로 넘겨야 할 때(4-3)
- **`OPTIONAL MATCH` 결과를 거를 때**(3-2 에서 이미 썼습니다. 거기서는 `WITH` 없이는 아예 걸러지지 않았죠)
- **`shortestPath` 로 구한 경로를 길이로 거를 때**(앞 시간 3-3 에서 미뤄 둔 것. 4-2 에서 잇습니다)

4-2 의 **계산한 값으로 거르기**는 이 넷과 성격이 조금 다릅니다. `WITH` 가 없으면 못 하는 일은 아니고(계산식을 `WHERE` 에 그대로 적어도 됩니다), **값에 이름을 한 번만 붙여 두고 여러 번 되쓰려고** 쓰는 것입니다.

여기서는 **파이프가 어떻게 흐르는지 눈에 보이도록** 일부러 단계를 나눠 씁니다. 실제 쿼리를 쓸 때는 "단계를 나눌 이유가 있나"를 먼저 물어보세요.

<img src="images/with_파이프라인.png" width="760">

*바로 아래 데모의 흐름: 식당 6곳에서 평점 조건으로 4곳, `WITH` 로 넘긴 뒤 가격(5만 원 이하) 조건으로 3곳이 남습니다.*

In [ ]:
# 평점 4.5 이상 -> 가격 5만원 이하 -> 평점 높은 순, 동점이면 이름순
# WITH r 은 살아남은 식당만 다음 절로 넘기는 파이프다.
# 정렬은 평점이 같을 때를 대비해 이름을 보조 정렬키로 하나 더 준다
rows = run_cypher("""
MATCH (r:Restaurant)
WHERE r.rating >= 4.5
WITH r
WHERE r.price <= 50000
RETURN r.name AS 식당, r.rating AS 평점, r.price AS 가격
ORDER BY r.rating DESC, r.name
""")
for r in rows:
    print(r['식당'], r['평점'], r['가격'])

In [ ]:
# DISTINCT 전달: 강남 식당들이 내는 '서로 다른' 요리 종류
# WITH DISTINCT 는 RETURN DISTINCT 와 달리 뒤에 WHERE 를 더 걸 수 있다
rows = run_cypher("""
MATCH (r:Restaurant)-[:LOCATED_IN]->(:Area {name:'강남'})
MATCH (r)-[:SERVES]->(c:Cuisine)
WITH DISTINCT c.name AS 요리
RETURN 요리
ORDER BY 요리
""")
print('강남 식당이 내는 요리 종류:', [r['요리'] for r in rows])

> 강남 식당 세 곳은 마침 요리가 서로 달라 이번에는 줄어든 줄이 없습니다. 같은 요리를 내는 곳이 여럿이면 `WITH DISTINCT` 가 그것들을 **한 줄로 줄여** 넘깁니다. 여기서는 중복을 없앤 결과를 그대로 돌려줬지만, `WITH` 로 넘겼으니 그 뒤에 `WHERE` 를 더 걸 수도 있습니다. 그게 `RETURN DISTINCT` 대신 `WITH DISTINCT` 를 쓰는 이유입니다.

### 🖐️ 함께 따라하기: 한 지역의 시설 종류

위 데모는 맛집의 **강남**이었습니다. 이번에는 **캠핑장 그래프**에서 **제주**에 있는 캠핑장들이 갖춘 **서로 다른 시설 종류**를 `WITH DISTINCT` 로 구해 출력하세요. 쿼리 모양은 같고 그래프가 다릅니다.

**확인 기준**: 시설 종류 **2가지**가 나옵니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) IN_REGION 으로 제주 지역 캠핑장을 잡는다
# 2) MATCH 를 한 줄 더 써서 그 캠핑장의 HAS_FACILITY 시설을 잇는다
# 3) WITH DISTINCT 로 시설 이름만 중복 없이 넘기고 RETURN 해 정렬 출력한다

### ✅ 바로 확인 퀴즈

**1.** `WITH` 와 `RETURN` 의 차이는 무엇인가요?

<details><summary>정답 보기</summary>

`RETURN` 은 쿼리를 **끝내며** 결과를 돌려주고, `WITH` 는 값을 골라 **다음 절로 넘깁니다**. 그래서 `WITH` 뒤에는 `WHERE`·`ORDER BY` 같은 절을 더 이어 쓸 수 있습니다.

</details>

**2.** 조건 두 개를 나란히 거는 것뿐이라면 `WITH` 로 나눠야 하나요?

<details><summary>정답 보기</summary>

**아닙니다.** 그냥 `WHERE A AND B` 한 줄이 낫습니다. `WITH` 는 중복 제거·중간 정렬·중간에 만든 값처럼 **결과를 한 번 손본 뒤 다시 걸러야 할 때** 필요합니다.

</details>

## 4-2. 값을 새로 만들어 넘기기

### 왜 필요할까요?
지금까지 `WITH` 는 **있는 값을 그대로** 넘겼습니다. 그런데 거르고 싶은 기준이 데이터에 **없는 값**일 때가 있습니다. "1인 가격은 있는데 **둘이 가면 얼마**인가", "정가는 있는데 **할인가**로 보면 어떤가". 이럴 때 `WITH` 에서 **계산해 이름을 붙여** 넘기고, 그다음 줄에서 그 이름으로 거릅니다.

### 문법: 표현식에는 `AS` 로 이름을 붙인다
```text
MATCH (r:Restaurant)
WITH r, r.price * 2 AS 둘이서      ← 계산한 값에 이름을 붙여 넘긴다
WHERE 둘이서 <= 60000              ← 붙인 이름으로 거른다
RETURN r.name AS 식당, 둘이서
```

> ⚠️ **`WITH` 에 표현식을 적을 때는 `AS` 가 필수입니다.** `WITH r.price` 라고만 쓰면 `Expression in WITH must be aliased (use AS)` 라는 문법 오류가 납니다. 넘길 값에 **부를 이름**이 없으면 다음 절에서 그 값을 가리킬 방법이 없기 때문입니다. 변수 하나를 그대로 넘길 때(`WITH r`)만 이름을 생략할 수 있습니다.

> 그리고 `WITH` 에 **적지 않은 것은 다음 절에서 사라집니다.** 위에서 `WITH r, ...` 처럼 `r` 을 함께 적은 이유가 그것입니다. 빼먹으면 `RETURN r.name` 에서 "`r` 이 뭐냐"는 오류가 납니다.

> **앞 시간에 미뤄 뒀던 자리가 여기입니다.** 3-3 에서 `shortestPath(...)` 뒤에 `WHERE length(p) >= 4` 를 붙이면 안 된다고 했습니다. Cypher 가 그것을 "구해 놓고 거른다" 가 아니라 **"4칸 이상이면서 가장 짧은 길을 찾아라"** 로 읽어, 조건에 맞는 것이 나올 때까지 경로를 계속 만들어 보기 때문이었죠. `WITH` 로 **한 단계 넘긴 뒤에** 걸면 그 일이 사라집니다.

```text
MATCH p = shortestPath( (a)-[:NEXT_TO*]-(b) )
WITH p, length(p) AS 홉수      ← 최단 경로를 먼저 구해 놓고 넘긴다
WHERE 홉수 >= 4                ← 구해 온 것을 거른다
RETURN 홉수
```

> **`ORDER BY` 만은 예외입니다.** `WITH`·`RETURN` 에 붙은 `ORDER BY` 는 아직 **넘기기 전 장면**을 보고 줄을 세우기 때문에, 고르지 않은 값으로도 정렬할 수 있습니다. 뒤에 나오는 `RETURN r.name AS 식당 ... ORDER BY r.rating DESC`(5-1) 가 그 모양입니다. 다만 **그 줄을 지나면** `r` 은 사라져서, 그다음 `MATCH` 나 `WHERE` 에서는 못 씁니다.

> 단 **`DISTINCT` 를 붙이면 이 예외가 사라집니다.** 4-1 의 `WITH DISTINCT c.name AS 요리` 뒤에 `ORDER BY r.rating` 을 붙이면 `In a WITH/RETURN with DISTINCT or an aggregation, it is not possible to access variables declared before the WITH/RETURN` 오류가 납니다. 중복을 없애는 순간 **남긴 값만 남고** 넘기기 전 장면은 볼 수 없기 때문입니다. 그럴 때는 정렬에 쓸 값도 `WITH` 에 함께 적어야 하는데, 그러면 **그 값까지 중복 판정에 들어가** 줄이 다시 늘 수 있습니다(요리는 넷인데 `r.rating` 을 같이 넘기면 평점이 다른 만큼 갈라집니다).

In [ ]:
# 데이터에 없는 '둘이서 가면 얼마' 를 만들어 넘기고 그 값으로 거른다
# 데이터에 없는 값이라 AS 로 이름을 붙여야 다음 줄에서 부른다.
# WITH 에 r 도 적어야 마지막 RETURN 에서 r.name 을 쓸 수 있다
rows = run_cypher("""
MATCH (r:Restaurant)
WITH r, r.price * 2 AS 둘이서
WHERE 둘이서 <= 60000
RETURN r.name AS 식당, r.price AS 일인분, 둘이서
ORDER BY 둘이서, 식당
""")
for r in rows:
    print(r['식당'], r['일인분'], '->', r['둘이서'])

> 여섯 곳 중 4곳이 남았습니다. **`둘이서` 라는 열은 데이터에 없습니다.** `WITH` 에서 만들어 이름을 붙였기 때문에 그다음 줄의 `WHERE` 와 마지막 `RETURN` 에서 모두 쓸 수 있었습니다. 이 조건 하나만이라면 `WHERE r.price * 2 <= 60000` 처럼 계산식을 그 자리에 그대로 적어도 답은 같습니다. `WITH` 의 값어치는 **계산식에 이름을 한 번만 붙여 두고 `WHERE` 와 `RETURN` 에서 되쓰는 것**입니다. 계산식이 길수록 차이가 커집니다.

위 문법 상자에서 말한 **`length(p)` 자리**도 실제로 돌려 봅시다. 3-3 에서 본 지민·서연은 스시효를 다리 삼아 이어져 있습니다. 그 최단 경로를 `WITH` 로 넘긴 뒤 홉수로 걸러 봅니다.

In [ ]:
# 앞 시간 3-3 에서 미뤄 둔 자리다. shortestPath 를 WITH 로 넘긴 뒤 길이로 거른다
# 기준을 파라미터로 받아 두 번 부른다. 한 번은 남고 한 번은 걸러지는 것을 보려고
def hops_at_least(n):
    """지민-서연 최단 경로를 구해 놓고, 홉수가 n 이상일 때만 그 홉수를 돌려준다."""
    return [r['홉수'] for r in run_cypher("""
    MATCH p = shortestPath( (a:Diner {name:'지민'})-[:VISITED*]-(b:Diner {name:'서연'}) )
    WITH p, length(p) AS 홉수
    WHERE 홉수 >= $n
    RETURN 홉수
    """, n=n)]


print('2 이상이면:', hops_at_least(2))
print('3 이상이면:', hops_at_least(3))

> 두 손님은 2홉으로 이어져 있어 기준이 2 일 때는 남고, 3 일 때는 빈 목록이 됩니다. **구해 놓은 경로를 놓고 거른 것**입니다. `WHERE 홉수 >= $n` 을 `WITH` 앞으로 옮기면, 앞 시간에 본 대로 Cypher 가 "그 길이 이상이면서 가장 짧은 길" 을 새로 찾아 나서 뜻이 달라집니다.

### 🖐️ 함께 따라하기: 1인당 가격으로 거르기

**캠핑장 그래프**의 `price` 는 **한 사이트(2인 기준) 가격**이라고 합시다. `WITH` 에서 **`c.price / 2 AS 인당가격`** 을 만들어 넘기고, **1인당 25,000원 이하**인 캠핑장만 남겨 이름·정가·인당가격을 인당가격 오름차순으로 출력하세요.

**확인 기준**: **3곳**이 나오고 맨 위는 **파도소리(인당 15000원)** 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Campsite) 로 캠핑장을 잡는다
# 2) WITH c, c.price / 2 AS 인당가격 으로 계산한 값에 이름을 붙여 넘긴다 (c 도 함께 넘긴다)
# 3) 다음 줄 WHERE 인당가격 <= 25000 으로 거른다
# 4) 이름·정가·인당가격을 RETURN 하고 인당가격 오름차순으로 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `WITH r.price * 2` 라고만 쓰면 어떻게 되나요?

<details><summary>정답 보기</summary>

**문법 오류**가 납니다(`Expression in WITH must be aliased (use AS)`). 계산한 값에는 **`AS 이름`** 을 붙여야 다음 절에서 그 값을 부를 수 있습니다.

</details>

**2.** `WITH r.price * 2 AS 둘이서` 라고만 쓰고 뒤에서 `RETURN r.name` 을 하면 어떻게 되나요?

<details><summary>정답 보기</summary>

`r` 을 **넘기지 않았으므로** 오류가 납니다. `WITH` 에 적은 것만 다음 절로 넘어갑니다. `WITH r, r.price * 2 AS 둘이서` 처럼 필요한 변수를 함께 적어야 합니다.

</details>

## 4-3. 중간에서 정렬해 자르기

### 왜 필요할까요?
4-1 의 두 데모는 사실 `AND` 나 `RETURN DISTINCT` 로도 되고, 4-2 의 계산값 필터도 계산식을 그대로 적으면 됩니다. `WITH` 가 **없으면 아예 못 하는** 일은 따로 있습니다. **중간에서 정렬해 몇 개만 남기고, 그 결과로 다음 단계를 잇는 것**입니다.

평점 상위 세 곳을 먼저 고른 뒤, 그 세 곳을 방문한 손님을 이어 봅니다. 자르는 시점이 어디냐에 따라 답이 달라집니다.

In [ ]:
# 평점순 3곳만 남긴 뒤 그 3곳의 방문 손님을 잇는다
# WITH 뒤의 ORDER BY ... LIMIT 은 '넘기기 전에' 자른다
rows = run_cypher("""
MATCH (r:Restaurant)
WITH r ORDER BY r.rating DESC, r.name LIMIT 3
MATCH (dn:Diner)-[:VISITED]->(r)
RETURN r.name AS 식당, r.rating AS 평점, dn.name AS 손님
ORDER BY 평점 DESC, 식당, 손님
""")
for r in rows:
    print(r['식당'], r['평점'], r['손님'])

In [ ]:
# 대비: 똑같은 조회에서 LIMIT 만 맨 뒤로 옮겨 본다
# 이번에는 다 이은 '뒤에' 자르므로 남는 것은 행 3줄이다
rows = run_cypher("""
MATCH (r:Restaurant)
MATCH (dn:Diner)-[:VISITED]->(r)
RETURN r.name AS 식당, r.rating AS 평점, dn.name AS 손님
ORDER BY 평점 DESC, 식당, 손님
LIMIT 3
""")
for r in rows:
    print(r['식당'], r['평점'], r['손님'])

> 위는 **4줄**, 아래는 **3줄**입니다. 쓴 문법은 같은데 **자르는 시점**이 달라서 묻는 것이 달라졌습니다. 위는 "평점 상위 3**곳**의 방문 전부", 아래는 "방문 목록의 앞 3**줄**" 입니다.

> 이 일은 `AND` 로 바꿔 쓸 수 없습니다. **먼저 줄 세워 자르고, 그 결과로 다음 단계를 잇는** 것은 `WITH` 만 할 수 있습니다. 다음 단원의 집계도 이 자리에서 이어집니다.

### 🖐️ 함께 따라하기: 평점 상위 두 곳에 묵은 캠퍼

**캠핑장 그래프**에서 **평점 상위 두 곳**을 먼저 자른 뒤, 그 두 곳에 묵은(`STAYED`) 캠퍼를 이으세요. `WITH c ORDER BY c.rating DESC, c.name LIMIT 2` 로 **먼저 자르고** 그다음 줄에서 `MATCH` 로 잇습니다.

**확인 기준**: **3행**이 나옵니다. 캠핑장은 **강가마루·별빛캠핑장** 두 곳뿐입니다.

> `LIMIT 2` 를 맨 뒤로 옮기면 답이 달라집니다. 그때는 "캠핑장 두 곳"이 아니라 "행 두 줄"이 됩니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Campsite) 로 캠핑장을 모두 잡는다
# 2) WITH c ORDER BY c.rating DESC, c.name LIMIT 2 로 상위 두 곳만 넘긴다
# 3) MATCH 를 한 줄 더 써서 (cp:Camper)-[:STAYED]->(c) 를 잇는다
# 4) 캠핑장·평점·캠퍼를 RETURN 하고 평점 내림차순, 캠핑장, 캠퍼 순으로 정렬해 출력한다

### ✅ 바로 확인 퀴즈

**1.** `WITH r ORDER BY ... LIMIT 3` 과, 맨 뒤에 `LIMIT 3` 을 붙이는 것은 무엇이 다른가요?

<details><summary>정답 보기</summary>

**자르는 시점**이 다릅니다. 앞엣것은 **다음 단계로 넘기기 전에** 잘라 "상위 3곳"을 정하고, 뒤엣것은 **다 이어 붙인 뒤** 잘라 "결과 3줄"을 남깁니다. 같은 문법으로 다른 질문을 하는 것입니다.

</details>

**2.** 이 일을 `WHERE ... AND ...` 로 바꿔 쓸 수 있나요?

<details><summary>정답 보기</summary>

**없습니다.** "상위 3곳"은 **다른 행들과 견줘 봐야** 정해지는 것이라 한 행만 보는 `WHERE` 로는 표현할 수 없습니다. 먼저 줄 세워 자르고 그 결과로 다음 단계를 잇는 것은 `WITH` 만 할 수 있습니다.

</details>

---
# 5. 정렬과 쪽 넘기기

4절이 **중간**에서 다듬는 법이었다면, 여기는 **마지막**에 결과를 내놓는 자리입니다.

- **5-1** `SKIP` 으로 그다음 쪽으로 넘어갑니다.
- **5-2** 동점이 쪽 넘기기를 어떻게 무너뜨리는지 봅니다.

## 5-1. 쪽 넘기기: `SKIP`

### 왜 필요할까요?
지난 시간에 `ORDER BY` 로 줄을 세우고 `LIMIT` 으로 위에서 몇 개만 남기는 법을 배웠습니다. 그런데 "그다음 몇 개" 를 보려면 한 가지가 더 필요합니다. 앞의 것을 **건너뛰는** `SKIP` 입니다.

| 절 | 뜻 |
|---|---|
| `ORDER BY`(지난 시간) | 줄을 세운다 |
| `LIMIT N`(지난 시간) | 앞에서 **N개만** 남긴다 |
| `SKIP N` | 앞의 **N개를 건너뛴다** |

둘을 함께 쓰면 **쪽 넘기기**가 됩니다. 1페이지는 `SKIP 0 LIMIT 3`, 2페이지는 `SKIP 3 LIMIT 3`, 3페이지는 `SKIP 6 LIMIT 3` 입니다. 순서는 `ORDER BY` → `SKIP` → `LIMIT` 로 적습니다.

> 최근 Cypher 는 `SKIP` 대신 **`OFFSET`** 이라고 적어도 받습니다(뜻은 똑같습니다). 문서나 남의 쿼리에서 만나면 같은 것으로 읽으세요. 우리 수업은 `SKIP` 으로 통일합니다.

> ⚠️ **`SKIP` 은 정렬이 확정돼야 뜻이 생깁니다.** 동점인데 보조 정렬키가 없으면 같은 행이 1·2페이지에 **두 번 나오거나** 어떤 행은 **영영 안 보일** 수 있습니다. `ORDER BY` 가 동점을 남기면 그 둘의 앞뒤는 **아무도 정해 주지 않기** 때문입니다. 쪽 넘기기를 할 때는 보조 정렬키가 **필수**입니다. 바로 아래에서 실제로 어긋나는 것을 봅니다.

<img src="images/skip_limit_페이징.png" width="760">

*정렬된 6행 위로 `SKIP 0 LIMIT 3`(1페이지)·`SKIP 3 LIMIT 3`(2페이지) 창이 미끄러집니다.*

In [ ]:
# 3곳씩 쪽 넘기기. 건너뛸 개수를 $skip 으로 넘긴다
def page(skip):
    """앞에서 skip 개를 건너뛴 뒤 식당 3곳을 평점 내림차순으로 돌려준다."""
    # 이름을 보조 정렬키로 둬 동점의 앞뒤를 매번 같게 만든다.
    # 절의 순서는 ORDER BY -> SKIP -> LIMIT 로 적는다
    return run_cypher("""
    MATCH (r:Restaurant)
    RETURN r.name AS 식당, r.rating AS 평점
    ORDER BY r.rating DESC, r.name
    SKIP $skip LIMIT 3
    """, skip=skip)

print('1페이지:', [(r['식당'], r['평점']) for r in page(0)])
print('2페이지:', [(r['식당'], r['평점']) for r in page(3)])

> 1페이지는 `['스시효', '비스트로홍', '딤섬각']`, 2페이지는 `['파스타부오노', '라멘야마', '국밥천국']` 입니다. 두 쿼리의 `ORDER BY` 가 **완전히 같고** 순서가 유일하기 때문에 페이지가 겹치지도, 빠지지도 않습니다.

### 🖐️ 함께 따라하기: 가격순 2페이지 뽑기

이번에는 **그래프도 정렬 기준도 바꿔** 봅니다. 위 데모는 맛집을 평점 내림차순으로 3곳씩이었지만, 여기서는 **캠핑장**을 **가격이 싼 순**으로 **2곳씩** 끊어 **2페이지(3~4번째로 싼 곳)** 를 뽑아 이름과 가격을 출력하세요.

`ORDER BY c.price` 로 오름차순 정렬하고 **보조 정렬키로 `c.name`** 을 붙인 뒤, `SKIP` 으로 앞의 두 곳을 건너뛰고 `LIMIT` 로 두 곳만 남깁니다.

**확인 기준**: **2곳**이 나오고, 가격은 45000원과 55000원입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 모든 캠핑장
# 2) RETURN 이름·가격, ORDER BY c.price(오름차순), c.name
# 3) SKIP 으로 앞 두 곳을 건너뛰고 LIMIT 로 두 곳만 남긴다 (= 2페이지)

### ✅ 바로 확인 퀴즈

**1.** `ORDER BY`·`SKIP`·`LIMIT` 은 어떤 순서로 적나요?

<details><summary>정답 보기</summary>

**`ORDER BY` → `SKIP` → `LIMIT`** 순서입니다. 먼저 줄을 세우고, 앞을 건너뛴 다음, 남길 개수를 자릅니다.

</details>

**2.** 한 페이지에 5개씩 보여 줄 때 **3페이지**를 뽑으려면 `SKIP`·`LIMIT` 에 얼마를 주나요?

<details><summary>정답 보기</summary>

`SKIP 10 LIMIT 5` 입니다. 앞의 두 페이지(5+5=10개)를 건너뛰고 5개를 남깁니다.

</details>

## 5-2. 동점이 쪽 넘기기를 무너뜨린다

### 왜 필요할까요?
이 데이터에는 평점 **4.6 인 식당이 두 곳** 있고, 그 둘이 하필 **3위와 4위**, 즉 페이지 경계에 놓입니다. "평점 내림차순"이라는 말만으로는 둘 중 누가 3위인지 정해지지 않습니다. 아래 두 쿼리는 **둘 다 평점 내림차순을 지키지만** 동점을 푸는 기준이 달라서, 1페이지와 2페이지가 어긋납니다. 말로만 듣지 말고 **실제로 무너지는 것**을 보겠습니다.

In [ ]:
# 동점을 이름순으로 푼 것과 비싼 순으로 푼 것을 견준다
# 1) 평점이 같으면 이름이 앞선 곳을 먼저 둔다
by_name = run_cypher("""
MATCH (r:Restaurant)
RETURN r.name AS 식당
ORDER BY r.rating DESC, r.name
SKIP 0 LIMIT 3
""")
# 2) 평점이 같으면 비싼 곳을 먼저 둔다. 3위와 4위가 뒤바뀐다
by_price = run_cypher("""
MATCH (r:Restaurant)
RETURN r.name AS 식당
ORDER BY r.rating DESC, r.price DESC
SKIP 3 LIMIT 3
""")
page1 = [r['식당'] for r in by_name]
page2 = [r['식당'] for r in by_price]
# 3) 두 페이지가 전체를 나눠 가졌는지 집합으로 대조한다
allnames = [r['식당'] for r in run_cypher("MATCH (r:Restaurant) RETURN r.name AS 식당")]
print('1페이지:', page1)
print('2페이지:', page2)
print('두 페이지에 겹쳐 나온 곳:', sorted(set(page1) & set(page2)))
print('어느 페이지에도 없는 곳:', sorted(set(allnames) - set(page1) - set(page2)))

> **딤섬각** 은 두 페이지에 겹쳐 나오고, **파스타부오노** 는 어느 페이지에도 안 나옵니다. 여섯 곳을 3곳씩 두 쪽으로 나눴는데 한 곳은 두 번 보이고 한 곳은 영영 안 보이는 것입니다. **정렬이 동점을 남기면 쪽 넘기기가 이렇게 무너집니다.** 그래서 두 페이지가 `ORDER BY` 를 **완전히 같게**, 그리고 **동점이 남지 않게**(이름 같은 유일한 값을 보조 정렬키로) 써야 합니다.

### 🖐️ 함께 따라하기: 동점을 넘어 나눠 보기

**캠핑장 그래프**에도 평점이 같은 곳이 두 군데 있습니다(**숲속카라반**·**노을언덕**, 둘 다 4.6). 평점 내림차순으로 **3곳씩** 두 페이지로 나누면 이 동점이 **페이지 경계에 걸립니다.**

보조 정렬키로 `c.name` 을 붙여 1·2페이지를 각각 뽑고, 파이썬 집합으로 **겹치는 곳이 없는지**와 **여섯 곳이 빠짐없이 나뉘었는지**를 확인하세요.

**확인 기준**: 겹치는 곳 **0곳**, 두 페이지를 합치면 **6곳** 입니다. 1페이지는 `['별빛캠핑장', '강가마루', '노을언덕']` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) page_of(skip) 함수를 만든다. ORDER BY c.rating DESC, c.name 에 SKIP $skip LIMIT 3
# 2) page_of(0)·page_of(3) 을 각각 뽑아 이름 목록으로 만든다
# 3) 두 목록의 교집합 크기와 합집합 크기를 출력해 겹침 0·전체 6 인지 확인한다

### ✅ 바로 확인 퀴즈

**1.** 두 페이지에 같은 행이 나오거나 어떤 행이 사라지는 이유는 무엇인가요?

<details><summary>정답 보기</summary>

`ORDER BY` 가 **동점을 남겨** 순서가 유일하지 않기 때문입니다. 동점끼리의 앞뒤는 아무도 정해 주지 않으므로 페이지마다 다르게 잡힐 수 있습니다.

</details>

**2.** 보조 정렬키로는 어떤 값을 골라야 하나요?

<details><summary>정답 보기</summary>

**값이 겹치지 않는 것**을 고릅니다. 이름이나 `id` 처럼 행마다 유일한 값을 마지막 정렬키로 붙이면 순서가 하나로 정해집니다. 그리고 페이지마다 `ORDER BY` 를 **완전히 같게** 써야 합니다.

</details>

---
## 🚀 응용 클론코딩: 추천 쿼리 조립

복합 쿼리는 **작게 시작해 한 조건씩 확장**합니다. 데모 도메인인 **맛집 그래프**로, 오늘 배운 것을 **한 함수에 엮어** 봅니다. 가고 싶은 지역 목록과 둘이서 쓸 예산을 받아 식당 두 곳을 추천하는 `recommend(areas, budget)` 을 완성하세요. 네 단계를 순서대로 쌓습니다.

1. 지역이 목록 안에 있는 식당만 (`WHERE a.name IN $areas`. 목록은 **파라미터로** 넘깁니다. 1-2)
2. `WITH` 에서 **`r.price * 2 AS 둘이서`** 를 만들어 예산 이하만 남기고 (4-2)
3. 평점 높은 순(동점이면 이름순)으로 **상위 두 곳만** 자른 뒤 (`WITH r ORDER BY ... LIMIT 2`. 4-3)
4. 그 두 곳의 셰프를 **`OPTIONAL MATCH`** 로 잇습니다 (셰프가 없는 곳도 빠뜨리지 않으려고. 3-1)

> 3번은 `AND` 로 바꿔 쓸 수 없습니다. 잘라 낼 대상을 **먼저** 정해야 하니까요. 4번을 그냥 `MATCH` 로 이으면 셰프 없는 식당이 통째로 사라져 **두 곳을 골라 놓고 한 곳만** 보게 됩니다.

**예시 결과**: `recommend(['홍대', '종로'], 60000)` 은 2행이고 1위는 `파스타부오노`(평점 4.6, 셰프 이부오노), 2위 `라멘야마` 은 셰프가 `None` 입니다. `recommend(['강남'], 100000)` 은 `비스트로홍`·`딤섬각` 두 곳이 나옵니다(스시효는 둘이서 16만 원이라 예산에서 걸립니다).

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) recommend(areas, budget) 함수를 정의한다
# 2) MATCH (r:Restaurant)-[:LOCATED_IN]->(a:Area) 에 WHERE a.name IN $areas 를 건다
# 3) WITH r, r.price * 2 AS 둘이서 로 넘기고 다음 줄 WHERE 둘이서 <= $budget 으로 거른다
# 4) WITH r ORDER BY r.rating DESC, r.name LIMIT 2 로 상위 두 곳만 넘긴다
# 5) OPTIONAL MATCH (ch:Chef)-[:WORKS_AT]->(r) 로 셰프를 (있으면) 잇는다
# 6) 식당·평점·셰프를 RETURN 하고, areas·budget 을 파라미터로 넘겨 두 번 호출해 출력한다

In [ ]:
# 🚀 응용 (같은 함수를 다른 조건으로 한 번 더 불러 보세요)
# 1) recommend(['강남'], 100000) 을 호출해 같은 형식으로 출력한다
# 2) 앞 호출과 견줘 어느 조건이 어느 식당을 떨어뜨렸는지 확인한다

---
## 참고: 패턴 안에 조건을 바로 적기

공식 문서를 보면 조건이 `WHERE` 절이 아니라 **패턴 괄호 안**에 들어가 있는 예제가 나옵니다. 최근 Cypher 에 더해진 표기로, **그 노드나 관계를 찾는 그 자리에서** 조건을 겁니다.

| 오늘 배운 표기 | 패턴 안에 적는 표기 |
|---|---|
| `MATCH (r:Restaurant) WHERE r.rating >= 4.5` | `MATCH (r:Restaurant WHERE r.rating >= 4.5)` |
| `MATCH (a)-[x:ROUTE]->(b) WHERE x.time < 60` | `MATCH (a)-[x:ROUTE WHERE x.time < 60]->(b)` |

결과는 같습니다. **가변길이 구간마다 조건을 걸어야 할 때** 특히 편합니다(`-[r:ROUTE WHERE r.time < 60]->{1,3}()` 처럼요). 다만 조건이 여럿이면 패턴 줄이 길어져 읽기 어려워지므로, **우리 수업은 `WHERE` 절에 모아 적는 쪽으로 통일합니다.** 문서에서 이 표기를 만났을 때 같은 뜻이라는 것만 알아 두세요.

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-1 | `IN`·`CONTAINS`·`STARTS WITH`·`ENDS WITH` | 목록 일치·부분 포함·접두·접미 일치 |
| 1-2 | `=~` · `IN $names` | 여러 패턴을 한 줄로. 목록은 이어 붙이지 말고 파라미터로 |
| 2-1 | `(a)-[:R1]->(b)-[:R2]->(c)` / `-[:R1\|R2]->` | 앞은 두 단계, 뒤는 한 단계에 종류 둘 |
| 2-2 | `WHERE (a)-[:R]->(b)` / `WHERE NOT (...)` | 관계의 유무 자체가 조건. 행이 늘지 않는다 |
| 2-3 | `EXISTS { ... WHERE ... }` | 상대 쪽에 조건을 걸어야 할 때. 안쪽 변수는 밖으로 못 나온다 |
| 3-1 | `OPTIONAL MATCH` | 짝이 없어도 행 유지(자리는 `null`). 관계형의 LEFT JOIN |
| 3-2 | `WITH` 뒤의 `WHERE` | `OPTIONAL MATCH` 바로 뒤 `WHERE` 는 못 거른다 |
| 3-3 | `OPTIONAL MATCH p = shortestPath(...)` + `p IS NOT NULL` | 닿는지 판별. 경로가 없어도 한 행이 온다 |
| 3-3 | `-[:R*]-` vs `-[:R*]->` | 화살표 하나가 질문의 뜻을 바꾼다 |
| 4-1 | `WITH r` / `WITH DISTINCT ...` | 값을 넘기며 거르고 중복을 없앤다 |
| 4-2 | `WITH r, 식 AS 이름` | 없는 값을 만들어 넘긴다. 표현식에는 `AS` 가 필수 |
| 4-3 | `WITH r ORDER BY ... LIMIT n` | 중간에서 잘라 다음 단계로. `AND` 로는 못 한다 |
| 5-1 | `ORDER BY` → `SKIP` → `LIMIT` | 정렬하고 건너뛰고 자른다 |
| 5-2 | 보조 정렬키 | 동점이 남으면 쪽 넘기기가 무너진다 |

- 복합 쿼리는 **작게 만들어 한 조건씩 확장**하면 안전합니다.
- 조건을 나란히 거는 것뿐이면 `AND` 한 줄로 충분합니다. `WITH` 는 중간에 결과를 손봐야 할 때 씁니다.
- 동점이 페이지 경계에 걸리면 쪽 넘기기가 실제로 무너집니다. 보조 정렬키로 순서를 유일하게 만드세요.
- 상한 없는 `*` 는 편하지만 큰 그래프에서는 비쌉니다. 상한을 두거나 `shortestPath` 로 감쌉니다.
- **방향 있는 관계는 화살표를 지켜야 의미가 맞습니다**. `-[:R*]-` 와 `-[:R*]->` 는 서로 다른 질문입니다.

## ⏭️ 예고: 다음 시간

지금까지는 결과를 **하나하나 나열**했습니다. 다음 시간에는 "몇 개인지", "합이 얼마인지", "평균은" 처럼 여러 행을 **하나로 요약**하는 집계와, 큰 그래프를 빠르게 찾도록 돕는 **인덱스**를 배웁니다.

수고하셨습니다!